# HW2 · Kaggle №2: One-stage Anchor-free (YOLO11s) + Query-based (RT-DETR) + Your Pipeline

| § | Пункт рубрики | Бали |
|---|---|---|
| 1 | One-stage Anchor-free: Y0, Y1, Y2, Y3 (tuned) | 1.00 |
| 2 | Query-based: D0, D1, D2, крива vs YOLO, щільні сцени | 0.75 |
| 3 | Your pipeline: P1 (дані), I_* (інференс), E1/E2 (ансамблі), S1/S2 (сабмішени) | 1.25 |

EDA, Faster R-CNN і RetinaNet робить ноутбук №1; його результати можна підтягнути в таблицю через `EXTRA_RES` (див. нижче).

## 0. Налаштування Google Colab
Перед запуском увімкни GPU: Runtime → Change runtime type → T4 GPU. Результати та checkpoints зберігаються на Google Drive.

**Kaggle access:** у Colab → Secrets створи секрет `KAGGLE_API_TOKEN`, встав API token, згенерований у Kaggle → Settings → API → Generate New Token, і дозволь доступ цьому ноутбуку.

In [2]:
# Google Colab setup
!pip -q install pycocotools ultralytics==8.4.201 ensemble-boxes "kaggle>=2.2.4"

from pathlib import Path
import os, zipfile, glob, shutil

# 1) Mount Google Drive: checkpoints/results will persist between sessions
from google.colab import drive
drive.mount('/content/drive')

COMP = 'multiple-object-video-detection-hw-2-ucu-ai-2026'
ROOT = Path('/content/drive/MyDrive/HW2_YOLO_RTDETR')
ROOT.mkdir(parents=True, exist_ok=True)

# Temporary working files and downloaded dataset stay on Colab's local disk (faster)
WORK = Path('/content/work')
WORK.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path('/content/competition_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 2) Kaggle API token authentication (Colab Secrets)
# In Colab: left sidebar -> Secrets -> add KAGGLE_API_TOKEN.
# Paste the token generated at Kaggle -> Settings -> API -> Generate New Token.
# Enable notebook access for this secret. Never print or share the token.
from google.colab import userdata

try:
    kaggle_api_token = userdata.get("KAGGLE_API_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Не знайдено Colab Secret KAGGLE_API_TOKEN. "
        "Додай його в Colab → Secrets і дозволь доступ цьому ноутбуку."
    ) from exc

if not kaggle_api_token:
    raise ValueError("Colab Secret KAGGLE_API_TOKEN порожній.")

os.environ["KAGGLE_API_TOKEN"] = kaggle_api_token
del kaggle_api_token

# 3) Download the competition dataset if it has not already been extracted.
# You must have joined the competition and accepted its rules on Kaggle first.
download_dir = DATA_ROOT / 'download'
download_dir.mkdir(exist_ok=True)
existing = list(DATA_ROOT.glob('*/train/annotations_coco.json'))
if not existing:
    !kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 -p /content/competition_data/download
    zips = list(download_dir.glob('*.zip'))
    if not zips:
        raise FileNotFoundError("Kaggle did not provide a ZIP. Check competition access and API credentials.")
    for z in zips:
        with zipfile.ZipFile(z) as archive:
            archive.extractall(DATA_ROOT)

# 4) Locate the expected dataset layout
hits = [p for p in DATA_ROOT.rglob('annotations_coco.json')
        if p.parent.name in ('train', 'val')
        and (p.parent.parent / 'test_images.csv').exists()
        and (p.parent.parent / 'train' / 'annotations_coco.json').exists()
        and (p.parent.parent / 'val' / 'annotations_coco.json').exists()]
if not hits:
    print("Contents of /content/competition_data:")
    for p in list(DATA_ROOT.iterdir())[:30]:
        print(" ", p)
    raise FileNotFoundError(
        "Dataset layout not found. Expected one folder containing train/annotations_coco.json, "
        "val/annotations_coco.json and test_images.csv. Check the downloaded archive."
    )
DATA = hits[0].parent.parent
EXTRA_RES = []  # optional: add Drive paths to teammates' results folders
print('DATA:', DATA, sorted(p.name for p in DATA.iterdir()))
print('Persistent ROOT:', ROOT, '| Temporary WORK:', WORK)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
100% 9.88G/9.88G [01:32<00:00, 115MB/s]

DATA: /content/competition_data ['download', 'test', 'test_images.csv', 'train', 'val']
Persistent ROOT: /content/drive/MyDrive/HW2_YOLO_RTDETR | Temporary WORK: /content/work


## 0.1 Протокол, дані, фолди, препроцесинг (спільне ядро, однакове в обох ноутбуках)

In [3]:
import json, pickle, shutil, time, math, gc, contextlib, io, logging, functools, random, subprocess
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, matplotlib.patches as patches
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from sklearn.model_selection import GroupKFold
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

CACHE, RES, RUNS, SUBS, TVCK = (ROOT / n for n in ['data', 'results', 'ckpt', 'submissions', 'ckpt_tv'])
for d in (CACHE, RES, RUNS, SUBS, TVCK): d.mkdir(parents=True, exist_ok=True)

# ---- ПРОТОКОЛ (спільний для всіх моделей; міняти лише разом з усіма) ----
SEED, STEP   = 42, 4          # кожен 4-й кадр кліпу (обґрунтування: EDA 1.3, IoU сусідніх вибраних кадрів ~0.66)
IMGSZ        = 960            # довга сторона входу
SAVE_MAX     = 1280           # кадри зберігаємо зменшеними до 1280 px (швидше читання, менше диска)
EPOCHS       = 8              # бюджет епох для КОЖНОГО експерименту
CLOSE_MOSAIC = 2
WARMUP_EPOCHS, WORKERS = 1, 4   # прогрів 1 епоха (дефолт Ultralytics 3 з'їв би майже половину бюджету)
VAL_CONF, MAX_DET, NMS_IOU = 0.01, 500, 0.6
N_FOLDS      = 3              # крос-валідація за кліпами (GroupKFold)
RUN_FOLDS    = [0, 1, 2]      # які фолди запускати; для налагодження: [0]
FRACTION     = 1.0            # частка кадрів для швидкої перевірки коду (0.02)
INNER_VAL    = 500            # кадрів у «внутрішній» валідації (лише криві по епохах)
CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕМАЄ')

GPU: Tesla T4


In [4]:
# ───── Анотації COCO → таблиці. iscrowd=0: цільові об'єкти; iscrowd=1: ignored regions. cls = category_id - 1 ─────
OFF = {'train': 0, 'val': 10**7}            # зсув image_id, щоб id з train і val не перетиналися (пулимо train+val)

def load_split(split):
    coco = json.load(open(DATA / split / 'annotations_coco.json'))
    imgs = pd.DataFrame(coco['images']).rename(columns={'id': 'image_id'})
    imgs['path'] = [str(DATA / split / 'images' / str(v) / Path(f).name) for v, f in zip(imgs.video, imgs.file_name)]
    imgs['split'] = split; imgs['image_id'] += OFF[split]
    a = pd.DataFrame(coco['annotations']); a['image_id'] += OFF[split]
    a[['x', 'y', 'w', 'h']] = pd.DataFrame(a.bbox.tolist(), index=a.index)
    a['cls'] = a.category_id - 1; a['split'] = split
    a = a.merge(imgs[['image_id', 'video', 'frame_id', 'width', 'height']], on='image_id')
    return imgs, a[a.iscrowd == 0].drop(columns=['bbox']).reset_index(drop=True), a[a.iscrowd == 1].drop(columns=['bbox']).reset_index(drop=True)

T = {s: load_split(s) for s in OFF}      # без кешу: шляхи залежать від місця монтування даних
imgs_all = pd.concat([T[s][0] for s in OFF], ignore_index=True)       # усі кадри train+val
obj_all  = pd.concat([T[s][1] for s in OFF], ignore_index=True)
ign_all  = pd.concat([T[s][2] for s in OFF], ignore_index=True)
assert not set(T['train'][0].video) & set(T['val'][0].video), 'кліпи train/val перетинаються!'
OBJ_BY_IMG = {k: g for k, g in obj_all.groupby('image_id')}
IGN_BY_IMG = {k: g[['x', 'y', 'w', 'h']].values for k, g in ign_all.groupby('image_id')}

# ───── Фолди: ВСІ кадри одного кліпу в одному фолді (GroupKFold за video) ─────
fold_of = {}
for k, (_, te) in enumerate(GroupKFold(N_FOLDS).split(imgs_all, groups=imgs_all.video)):
    fold_of.update({v: k for v in imgs_all.video.iloc[te].unique()})
pd.Series(fold_of).to_csv(CACHE / 'folds.csv')

FR = imgs_all.sort_values(['video', 'frame_id'])
FR = FR[FR.groupby('video').cumcount() % STEP == 0].copy()           # підвибірка: кожен STEP-й кадр
FR['fold'] = FR.video.map(fold_of)
if FRACTION < 1: FR = FR.sample(frac=FRACTION, random_state=SEED)
FR = FR.reset_index(drop=True)

TRAIN = lambda k: FR[FR.fold != k]                                   # train-частина фолду k
VAL   = lambda k: FR[FR.fold == k]                                   # out-of-fold валідація (кліпи, яких модель не бачила)
VSUB  = lambda k: VAL(k).sample(min(INNER_VAL, len(VAL(k))), random_state=SEED)   # мала підвибірка для кривих по епохах
print(FR.groupby('fold').agg(clips=('video', 'nunique'), frames=('image_id', 'size')))

      clips  frames
fold               
0        21    2263
1        21    2265
2        21    2258


In [5]:
def prep_all(root, mask):
    """Копія кадрів FR у root: картинка (<=SAVE_MAX px, ignored regions замальовані сірим якщо mask) + YOLO-мітка з наших таблиць."""
    root = Path(root)
    def one(r):
        img = root / 'images' / r.video / Path(r.path).name; lbl = root / 'labels' / r.video / (img.stem + '.txt')
        if img.exists(): return
        img.parent.mkdir(parents=True, exist_ok=True); lbl.parent.mkdir(parents=True, exist_ok=True)
        g = OBJ_BY_IMG.get(r.image_id); rows = []
        if g is not None:
            b = np.clip(np.c_[g.x, g.y, g.x + g.w, g.y + g.h], 0, [r.width, r.height] * 2); ok = (b[:, 2] > b[:, 0]) & (b[:, 3] > b[:, 1])
            rows = [f'{c} {(x1+x2)/2/r.width:.6f} {(y1+y2)/2/r.height:.6f} {(x2-x1)/r.width:.6f} {(y2-y1)/r.height:.6f}'
                    for c, (x1, y1, x2, y2) in zip(g.cls.values[ok], b[ok])]
        lbl.write_text('\n'.join(rows))
        im = Image.open(r.path).convert('RGB')
        if mask:
            a = np.array(im)
            for x, y, w, h in IGN_BY_IMG.get(r.image_id, []): a[int(y):int(np.ceil(y + h)), int(x):int(np.ceil(x + w))] = 114
            im = Image.fromarray(a)
        im.thumbnail((SAVE_MAX, SAVE_MAX)); im.save(img, quality=95)       # картинку пишемо останньою: вона маркер «готово»
    with ThreadPoolExecutor(8) as ex: list(ex.map(one, list(FR.itertuples())))
    print('кадрів підготовлено:', len(FR), '->', root)

def fold_yaml(k, root=None):
    """Списки кадрів для фолду k (train = інші фолди, val = INNER_VAL кадрів фолду k) + yaml для Ultralytics."""
    root = Path(root or WORK)
    for name, d in (('train', TRAIN(k)), ('val', VSUB(k))):
        (root / f'f{k}_{name}.txt').write_text('\n'.join(str(root / 'images' / r.video / Path(r.path).name) for r in d.itertuples()))
    y = root / f'data_f{k}.yaml'
    y.write_text(f"path: {root}\ntrain: {root}/f{k}_train.txt\nval: {root}/f{k}_val.txt\nnames: {dict(enumerate(CLASSES))}\n")
    return y

prep_all(WORK, mask=True)

кадрів підготовлено: 6786 -> /content/work


In [6]:
# ───── оцінка: ОДНА функція для всіх моделей (COCO mAP на кадрах фолду, ignored regions відкидаються) ─────
_GT = {}
def _gt():
    if not _GT:
        o = obj_all[obj_all.image_id.isin(FR.image_id)]
        g = dict(images=[dict(id=int(i), width=int(w), height=int(h)) for i, w, h in zip(FR.image_id, FR.width, FR.height)],
                 annotations=[dict(id=n + 1, image_id=int(i), category_id=int(c) + 1, bbox=[float(x), float(y), float(w), float(h)], area=float(w * h), iscrowd=0)
                              for n, (i, c, x, y, w, h) in enumerate(zip(o.image_id, o.cls, o.x, o.y, o.w, o.h))],
                 categories=[dict(id=i + 1, name=c) for i, c in enumerate(CLASSES)])
        cg = COCO(); cg.dataset = g
        with contextlib.redirect_stdout(io.StringIO()): cg.createIndex()
        _GT['coco'] = cg
    return _GT['coco']

def drop_in_ignore(xyxy, sc, cl, ign, thr=0.5):
    if len(xyxy) == 0 or len(ign) == 0: return xyxy, sc, cl
    gx1, gy1, gx2, gy2 = ign[:, 0], ign[:, 1], ign[:, 0] + ign[:, 2], ign[:, 1] + ign[:, 3]
    ix = (np.minimum(xyxy[:, None, 2], gx2) - np.maximum(xyxy[:, None, 0], gx1)).clip(0)
    iy = (np.minimum(xyxy[:, None, 3], gy2) - np.maximum(xyxy[:, None, 1], gy1)).clip(0)
    area = ((xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])).clip(1e-6)
    keep = (ix * iy).sum(1) / area < thr
    return xyxy[keep], sc[keep], cl[keep]

def coco_eval(preds, v, ms=float('nan')):
    """preds: {шлях: (xyxy, score, cls0)}; v: кадри. -> mAP, AP50, AP_small/medium/large, мс/кадр."""
    rows = []
    for iid, p in zip(v.image_id, v.path):
        xyxy, sc, cl = drop_in_ignore(*preds[str(Path(p))], IGN_BY_IMG.get(iid, np.zeros((0, 4))))
        if len(sc): rows.append(np.c_[np.full(len(sc), iid), xyxy[:, 0], xyxy[:, 1], xyxy[:, 2] - xyxy[:, 0], xyxy[:, 3] - xyxy[:, 1], sc, cl + 1])
    dets = np.concatenate(rows).astype(np.float64) if rows else np.zeros((0, 7))
    cg = _gt()
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(cg, cg.loadRes(dets), 'bbox'); E.params.imgIds = [int(i) for i in v.image_id]
        E.params.maxDets = [1, 10, MAX_DET]; E.evaluate(); E.accumulate(); E.summarize()
    P = E.eval['precision']                  # [IoU, recall, клас, площа(all/small/medium/large), maxDets]; беремо maxDets=MAX_DET
    def ap(area=0, iou=None):
        p = P[:, :, :, area, 2] if iou is None else P[[int(np.argmin(abs(E.params.iouThrs - iou)))], :, :, area, 2]
        return float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    return dict(mAP=ap(), AP50=ap(iou=0.5), AP_small=ap(1), AP_medium=ap(2), AP_large=ap(3), ms_per_img=ms)

def evaluate(pred_fn, v):
    preds, ms = pred_fn([str(p) for p in v.path])
    out = coco_eval(preds, v, ms); del preds; gc.collect(); return out

def make_submission(pred_fn, out_csv):
    test = pd.read_csv(DATA / 'test_images.csv'); part = Path(str(out_csv) + '.part'); done_ = {}
    if part.exists(): done_ = dict(pd.read_csv(part, dtype=str, keep_default_na=False).values)
    for clip, g in test[~test.image_id.isin(done_)].groupby('clip'):
        preds, _ = pred_fn([str(DATA / f) for f in g.file_name])
        for iid, f in zip(g.image_id, g.file_name):
            x, s, c = preds[str(DATA / f)]
            done_[iid] = ' '.join(f'{k} {p:.4f} {a:.1f} {b:.1f} {x2-a:.1f} {y2-b:.1f}' for (a, b, x2, y2), p, k in zip(x, s, c))
        pd.DataFrame(list(done_.items()), columns=['image_id', 'PredictionString']).to_csv(part, index=False)
    pd.DataFrame({'image_id': test.image_id, 'PredictionString': [done_[i] for i in test.image_id]}).to_csv(out_csv, index=False)
    print('збережено', out_csv, '| рядків:', len(test))

# ───── облік експериментів: results/<exp>_f<k>.json  ─────
def done(name): return (RES / f'{name}.json').exists()
def save_result(name, metrics, cfg, extra=None):
    json.dump({'metrics': {k: float(v) for k, v in metrics.items()}, 'cfg': cfg, **(extra or {})}, open(RES / f'{name}.json', 'w'), indent=2, default=str)
    print(name, {k: round(float(v), 4) for k, v in metrics.items()})

def summary():
    """Середнє по фолдах (out-of-fold оцінка) + std mAP. Читає результати всіх трьох ноутбуків."""
    rows = []
    for f in sorted(p for d in [RES, *EXTRA_RES] for p in Path(d).glob('*_f[0-9].json')):
        e, k = f.stem.rsplit('_f', 1); rows.append({'exp': e, 'fold': int(k), **json.load(open(f))['metrics']})
    if not rows: return pd.DataFrame()
    g = pd.DataFrame(rows).groupby('exp'); out = g[['mAP', 'AP50', 'AP_small', 'AP_medium', 'AP_large', 'ms_per_img']].mean().round(4)
    out['mAP_std'] = g.mAP.std().round(4); out['n_folds'] = g.size(); return out

def better(exp, base, margin=0.002):
    """True, якщо exp дав більший mean mAP по фолдах, ніж base (з запасом на шум). Так обираємо складові «tuned» моделі."""
    s = summary(); return exp in s.index and base in s.index and s.loc[exp, 'mAP'] > s.loc[base, 'mAP'] + margin

In [7]:
from ultralytics import YOLO, RTDETR, settings
settings.update({'wandb': False})
logging.getLogger('ultralytics').addFilter(type('F', (logging.Filter,), {'filter': lambda s, r: 'half' not in r.getMessage()})())

def yolo_pred(model, imgsz, conf=VAL_CONF, iou=NMS_IOU, augment=False, bs=None):
    bs = bs or (8 if imgsz <= 960 else 4)
    def fn(paths):
        out, t = {}, 0.
        for i in range(0, len(paths), bs):
            for r in model.predict(paths[i:i+bs], imgsz=imgsz, conf=conf, iou=iou, augment=augment, max_det=MAX_DET, half=True, verbose=False, device=0):
                b = r.boxes; out[str(Path(r.path))] = (b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)); t += sum(r.speed.values())
        torch.cuda.empty_cache(); return out, t / max(len(paths), 1)
    return fn

Wt = lambda exp, k: RUNS / f'{exp}_f{k}' / 'weights' / 'last.pt'

def run_det(exp, k, Net=YOLO, weights='yolo11s.pt', imgsz=IMGSZ, batch=16, data=None, nq=None, **kw):
    """Ultralytics-тренування на фолді k + оцінка ОСТАННЬОЇ епохи нашою evaluate на всьому out-of-fold val. Відновлюється з last.pt."""
    name = f'{exp}_f{k}'
    if done(name): print('Вже зроблено:', name); return
    d = RUNS / name / 'weights'; y = data or fold_yaml(k)      # списки кадрів відновлюємо завжди: після нової сесії WORK порожній
    if (d / 'last.pt').exists():
        try:
            m = Net(str(d / 'last.pt')); m.train(resume=True)
        except Exception as e: print('resume пропущено:', e)
    else:
        m = Net(weights)
        if nq:                                   # кількість queries RT-DETR: виставляємо у моделі та її EMA-копії після побудови
            def _set_nq(tr):
                for mod in (tr.model, getattr(getattr(tr, 'ema', None), 'ema', None)):
                    if mod is not None: mod.model[-1].num_queries = nq
            m.add_callback('on_train_start', _set_nq)
        m.train(data=str(y), imgsz=imgsz, epochs=EPOCHS, batch=batch, seed=SEED, project=str(RUNS), name=name, exist_ok=True, device=0, amp=True,
                workers=WORKERS, plots=True, patience=100, close_mosaic=CLOSE_MOSAIC, warmup_epochs=WARMUP_EPOCHS, **kw)
    gc.collect(); torch.cuda.empty_cache()
    save_result(name, evaluate(yolo_pred(Net(str(d / 'last.pt')), imgsz), VAL(k)),
                dict(model=weights, imgsz=imgsz, epochs=EPOCHS, batch=batch, fold=k, nq=nq, **kw), {'results_csv': str(RUNS / name / 'results.csv')})

def curve_mean(exp):
    """Крива val-метрик Ultralytics по епохах, усереднена по доступних фолдах."""
    dfs = []
    for k in RUN_FOLDS:
        f = RUNS / f'{exp}_f{k}' / 'results.csv'
        if f.exists(): df = pd.read_csv(f); df.columns = [c.strip() for c in df.columns]; dfs.append(df)
    return pd.concat(dfs).groupby('epoch').mean() if dfs else None

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 1. One-stage Anchor-free: YOLO11s (1.00)
Рубрика: *той самий протокол; тюнінг ≥2 параметрів; як призначаються позиції без anchors і вплив на дрібні та щільні об'єкти; діапазон пошуку, вибрані налаштування, результати; порівняння з anchor-based за розмірами об'єктів (колонки AP_small/medium/large в `summary()`).*

**Налаштування:** `yolo11s.pt` (COCO), `IMGSZ`, `EPOCHS`, `close_mosaic`, `optimizer=auto` (для цього розміру датасету це AdamW), AMP, `seed=42`.

| id | Що змінюємо | Діапазон |
|---|---|---|
| Y0 | baseline | `imgsz=960`, `box=7.5` |
| Y1 | роздільність | `imgsz` ∈ {960, **1280**} (`batch=8`) |
| Y2 | вага box-лоссу | `box` ∈ {7.5, **10**} |
| Y3 | **tuned**: комбінація тих із Y1/Y2, що покращили Y0 (автоматично, поріг +0.002 mAP) | вибрані налаштування |

**Застереження для звіту:** у Ultralytics втрата боксу це CIoU + DFL; замінити її без правок бібліотеки не можна, тож «зміна лоссу» тут означає зміну **ваги** box-складової.

**Записати у звіт:** як призначаються позиції без anchors (task-aligned assigner: для кожного GT вибираються позиції у центрі GT з найвищим поєднанням classification score та IoU); вплив на дрібні об'єкти (на 960 px об'єкт 10 px займає <2 клітинок stride-8 карти, мало кандидатів) і щільні (кілька GT конкурують за ті самі позиції).

In [ ]:
# Y0: baseline
for k in RUN_FOLDS: run_det('Y0_yolo11s_960', k, imgsz=960, batch=16)

In [ ]:
# Y1: роздільність 1280
for k in RUN_FOLDS: run_det('Y1_yolo11s_1280', k, imgsz=1280, batch=8)

In [ ]:
# Y2: вага box-лоссу 10
for k in RUN_FOLDS: run_det('Y2_yolo11s_960_box10', k, imgsz=960, batch=16, box=10.0)

In [ ]:
# Y3: tuned = складові Y1/Y2, що дали приріст mAP над Y0 (out-of-fold, поріг +0.002)
u1, u2 = better('Y1_yolo11s_1280', 'Y0_yolo11s_960'), better('Y2_yolo11s_960_box10', 'Y0_yolo11s_960')
sz = 1280 if u1 else 960; kw3 = dict(box=10.0) if u2 else {}; print('у tuned входять: imgsz 1280 =', u1, '| box=10 =', u2)
if u1 or u2:
    for k in RUN_FOLDS: run_det('Y3_yolo11s_tuned', k, imgsz=sz, batch=8 if sz > 960 else 16, **kw3)
else: print('жоден варіант не покращив Y0: tuned = baseline')

In [ ]:
# Діапазон пошуку, вибрані налаштування, результати (out-of-fold, mean по фолдах) для звіту
tab = summary(); print(tab.loc[[e for e in ['Y0_yolo11s_960', 'Y1_yolo11s_1280', 'Y2_yolo11s_960_box10', 'Y3_yolo11s_tuned'] if e in tab.index]].to_string())

## 2. Query-based: RT-DETR-L (0.75)
Рубрика: *порівнянний бюджет; тюнінг (queries відносно щільності сцени, matching cost, denoising, окремі lr); як bipartite matching прибирає NMS; графік val-mAP по епохах проти CNN-детектора; дрібні об'єкти й щільні сцени; якщо не збіглося, сказати про це.*

**Бюджет:** ті самі `IMGSZ`, `EPOCHS`, `close_mosaic`, фолди, кадри й GPU, що й у YOLO; `rtdetr-l.pt`, `batch=4`. Оптимізатор для всіх D-експериментів **явно** AdamW, `lr0=1e-4`, `weight_decay=1e-4` (авто-lr Ultralytics ≈ 8e-4 завеликий для трансформера).

| id | Що змінюємо |
|---|---|
| D0 | baseline: 300 queries, `lr0=1e-4` |
| D1 | кількість queries 100 (середня щільність ~45, p95 = 111, max = 180: 100 менше за найщільніші кадри) |
| D2 | `lr0=2e-4` (діапазон {1e-4, 2e-4}) |

**Обмеження Ultralytics (для звіту чесно):** вага matching cost (class/bbox/giou) і окремий lr для backbone у API не виставляються; denoising увімкнений за замовчуванням. Тому тюнимо кількість queries і lr.

**Перевірка D1:** після тренування має надрукуватись `num_queries: 100`. Якщо 300, callback не спрацював (механізм не перевірявся на GPU), D1 недійсний і значення треба задавати правкою yaml моделі.

**Записати у звіт:** bipartite matching (угорський алгоритм) ставить у відповідність кожному GT рівно одну query, решта вчаться передбачати «немає об'єкта», тому дублікатів немає і NMS не потрібен; ціна: повільніша збіжність, бо на ранніх епохах призначення нестабільне (для цього існує denoising). Якщо крива mAP ще росте в кінці 8 епох, пряма заява «не збіглося за бюджет» і опис спостережень.

In [ ]:
# D0: baseline (300 queries, lr0=1e-4)
for k in RUN_FOLDS: run_det('D0_rtdetr_l_960', k, Net=RTDETR, weights='rtdetr-l.pt', imgsz=IMGSZ, batch=4, optimizer='AdamW', weight_decay=1e-4, lr0=1e-4)

New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/data_f0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/8      6.33G      1.848     0.3028     0.3759        106        960: 100% ━━━━━━━━━━━━ 1131/1131 1.8it/s 10:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.6it/s 24.0s
                   all        500      24857      0.455      0.176     0.0781      0.034

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/8      6.77G      1.135     0.5908     0.1757        195        960: 0% ──────────── 0/1131  0.6s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/8      6.77G      1.174     0.5385     0.1653        241        960: 100% ━━━━━━━━━━━━ 1131/1131 1.8it/s 10:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.9it/s 22.1s
                   all        500      24857       0.58      0.193      0.075     0.0344

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/8      6.77G     0.8048     0.8608     0.1302         77        960: 0% ──────────── 0/1131  0.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (39 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        7/8      7.01G      0.635     0.7446    0.07381        141        960: 100% ━━━━━━━━━━━━ 1132/1132 1.8it/s 10:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.7it/s 23.1s
                   all        500      24170      0.179      0.378      0.154     0.0838

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        8/8      7.01G      0.679     0.6964    0.07151        155        960: 0% ──────────── 0/1132  0.6s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        8/8      7.01G     0.6225     0.7313     0.0715        201        960: 100% ━━━━━━━━━━━━ 1132/1132 1.8it/s 10:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.7it/s 23.0s
                   all        500      24170      0.183      0.383      0.159     0.0883

8 epochs completed in 1.506 hours.
Optimizer stripped from /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D0_rtdetr_l_960_f2/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D0_rtdetr_l_960_f2/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D0_rtdetr_l_960_f2/weights/best.pt...
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.5it/s 25.4s
          

In [ ]:
# D1: 100 queries
for k in RUN_FOLDS: run_det('D1_rtdetr_l_960_nq100', k, Net=RTDETR, weights='rtdetr-l.pt', imgsz=IMGSZ, batch=4, optimizer='AdamW', weight_decay=1e-4, lr0=1e-4, nq=100)
print('num_queries (D1):', RTDETR(str(Wt('D1_rtdetr_l_960_nq100', RUN_FOLDS[0]))).model.model[-1].num_queries)

New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/data_f0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/8      6.02G      1.679     0.3574     0.4683        106        960: 100% ━━━━━━━━━━━━ 1131/1131 1.9it/s 9:53
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.8it/s 22.3s
                   all        500      24857      0.535      0.246      0.134     0.0762

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/8      6.47G      1.255     0.4703     0.2468        195        960: 0% ──────────── 0/1131  0.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/8      6.47G       1.26     0.4684     0.3042        241        960: 100% ━━━━━━━━━━━━ 1131/1131 1.9it/s 9:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 3.1it/s 20.0s
                   all        500      24857      0.267      0.354      0.168     0.0951

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/8      6.47G      1.039     0.6841     0.2554         77        960: 0% ──────────── 0/1131  0.6s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (39 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        7/8      7.16G      2.234  1.846e-06     0.4613        141        960: 100% ━━━━━━━━━━━━ 1132/1132 1.9it/s 9:56
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 3.2it/s 19.4s
                   all        500      24170          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        8/8      7.16G      2.335  1.521e-06     0.4761        155        960: 0% ──────────── 0/1132  0.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        8/8      7.16G      2.218  1.905e-06     0.4499        201        960: 100% ━━━━━━━━━━━━ 1132/1132 1.9it/s 9:48
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 3.2it/s 19.6s
                   all        500      24170          0          0          0          0

8 epochs completed in 1.384 hours.
Optimizer stripped from /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D1_rtdetr_l_960_nq100_f2/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D1_rtdetr_l_960_nq100_f2/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D1_rtdetr_l_960_nq100_f2/weights/best.pt...
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 101.9 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 3.0it/s

In [8]:
# D2: lr0=2e-4
for k in RUN_FOLDS: run_det('D2_rtdetr_l_960_lr2e4', k, Net=RTDETR, weights='rtdetr-l.pt', imgsz=IMGSZ, batch=4, optimizer='AdamW', weight_decay=1e-4, lr0=2e-4)

New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/data_f0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/8      6.33G      1.931     0.2906      0.386        106        960: 100% ━━━━━━━━━━━━ 1131/1131 1.8it/s 10:20
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.6it/s 24.7s
                   all        500      24857      0.407      0.188      0.104     0.0495

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/8      6.77G     0.9446     0.7027     0.1636        195        960: 0% ──────────── 0/1131  0.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/8      6.77G     0.9191     0.6759     0.1299        241        960: 100% ━━━━━━━━━━━━ 1131/1131 1.8it/s 10:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.8it/s 22.7s
                   all        500      24857      0.219      0.397     0.0963     0.0491

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/8      6.77G     0.6907     0.9449     0.1013         77        960: 0% ──────────── 0/1131  0.6s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (39 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        7/8      7.01G     0.6005     0.6857    0.06737        141        960: 100% ━━━━━━━━━━━━ 1132/1132 1.8it/s 10:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.7it/s 23.2s
                   all        500      24170      0.248      0.393      0.228      0.123

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        8/8      7.01G     0.6431      0.654    0.06344        155        960: 0% ──────────── 0/1132  0.5s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:186.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        8/8      7.01G     0.5987     0.6304     0.0663        201        960: 100% ━━━━━━━━━━━━ 1132/1132 1.8it/s 10:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.7it/s 23.2s
                   all        500      24170      0.483      0.433       0.38      0.205

8 epochs completed in 1.509 hours.
Optimizer stripped from /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D2_rtdetr_l_960_lr2e4_f2/weights/last.pt, 66.3MB
Optimizer stripped from /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D2_rtdetr_l_960_lr2e4_f2/weights/best.pt, 66.3MB

Validating /content/drive/MyDrive/HW2_YOLO_RTDETR/ckpt/D2_rtdetr_l_960_lr2e4_f2/weights/best.pt...
Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,000,180 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 2.5it/

## 3. YOLO (CNN) vs RT-DETR (query-based): криві, швидкість, дрібні об'єкти, щільні сцени
Метрика Ultralytics однакова для обох (на `INNER_VAL` кадрах, усереднено по фолдах), тож криві порівнянні; обидві моделі мають однаковий бюджет епох. Година тренування береться з `results.csv`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for exp, lab in [('Y0_yolo11s_960', 'YOLO11s Y0 (CNN)'), ('Y3_yolo11s_tuned', 'YOLO11s Y3 tuned'), ('D0_rtdetr_l_960', 'RT-DETR-L D0 (300 q)'),
                 ('D1_rtdetr_l_960_nq100', 'RT-DETR-L D1 (100 q)'), ('D2_rtdetr_l_960_lr2e4', 'RT-DETR-L D2 (lr 2e-4)')]:
    c = curve_mean(exp)
    if c is not None:
        ax[0].plot(c.index, c['metrics/mAP50-95(B)'], marker='o', label=lab); ax[1].plot(c.index, c['metrics/mAP50(B)'], marker='o', label=lab)
        print(lab, '| час одного фолду, год:', round(c['time'].iloc[-1] / 3600, 2), '| mAP50-95 на останній епосі:', round(c['metrics/mAP50-95(B)'].iloc[-1], 4),
              '| приріст за останні 2 епохи:', round(c['metrics/mAP50-95(B)'].iloc[-1] - c['metrics/mAP50-95(B)'].iloc[-3], 4))
for a, t in zip(ax, ['val mAP50-95', 'val mAP50']): a.set_title(t); a.set_xlabel('епоха'); a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(CACHE / 'curve_yolo_vs_rtdetr.png', dpi=110); plt.show()

In [ ]:
# AP по терцилях кількості GT на кадр (фолд k0; одні й ті самі кадри для обох моделей): дрібні об'єкти й щільні сцени
k0 = RUN_FOLDS[0]; v = VAL(k0); tab = summary()
cnt = np.array([len(OBJ_BY_IMG.get(i, [])) for i in v.image_id])
terc = pd.qcut(pd.Series(cnt).rank(method='first'), 3, labels=['рідкі', 'середні', 'щільні']).values
def density_table(pred_fn):
    preds, _ = pred_fn([str(p) for p in v.path]); out = {}
    for g in ['рідкі', 'середні', 'щільні']:
        m = coco_eval(preds, v[terc == g]); out[g] = {'mAP': m['mAP'], 'AP_small': m['AP_small']}
    del preds; return pd.DataFrame(out).T.round(4)
Yc = [e for e in tab.index if e.startswith('Y')]; Dc = [e for e in tab.index if e.startswith('D')]
yb = max(Yc, key=lambda e: tab.loc[e, 'mAP']); db = max(Dc, key=lambda e: tab.loc[e, 'mAP'])
cfgsz = lambda e: json.load(open(RES / f'{e}_f{k0}.json'))['cfg']['imgsz']
print('кращі за mAP:', yb, '|', db)
print(pd.concat({yb: density_table(yolo_pred(YOLO(str(Wt(yb, k0))), cfgsz(yb))), db: density_table(yolo_pred(RTDETR(str(Wt(db, k0))), cfgsz(db)))}, axis=1))
print('межі терцилів (GT/кадр):', np.percentile(cnt, [33.3, 66.7]).round(0))

## 4. Your pipeline (1.25)
Рубрика: *довести пайплайн до найкращого результату через дані, інференс і моделювання; **окремо показати ефект даних/інференсу й ефект моделі**; послідовна валідація; вибір 2 сабмішенів за внутрішньою оцінкою.*

| Блок | Експеримент | Тип ефекту |
|---|---|---|
| 4.1 | P1: без маскування ignored regions (решта як у Y0) | **дані** |
| 4.2 | I_*: роздільність, NMS `iou`, flip-TTA на готових вагах кращого YOLO | **інференс** (без перенавчання) |
| 4.3 | Y1-Y3, D0-D2 з розділів 1-2 | **модель** |
| 4.4 | E1: WBF двох YOLO; E2: WBF YOLO + RT-DETR (на кожному фолді, лише моделі, що не бачили цей фолд) | інференс/модель |
| 4.5 | S1, S2: сабмішени = WBF ансамбль fold-моделей | |

Різниці менші за `mAP_std` з `summary()` вважати шумом. Два сабмішени обирай за out-of-fold mAP з таблиці, а не за публічним скором.

In [ ]:
# 4.1 абляція маскування (дані): окрема копія без маскування, решта як у Y0
WORK_NM = WORK.parent / 'work_nomask'; prep_all(WORK_NM, mask=False)
for k in RUN_FOLDS: run_det('P1_yolo11s_960_nomask', k, imgsz=960, batch=16, data=fold_yaml(k, WORK_NM))

In [ ]:
# 4.2 інференс-експерименти на вагах кращого YOLO (кожен фолд зі своїми вагами). Кращий за OOF mAP серед Y0-Y3:
tab = summary(); YC = [e for e in tab.index if e.startswith('Y')]
ranked = sorted(YC, key=lambda e: -tab.loc[e, 'mAP']); BEST, SECOND = ranked[0], (ranked[1] if len(ranked) > 1 else ranked[0])
cfg_of = lambda e: json.load(open(RES / f'{e}_f{RUN_FOLDS[0]}.json'))['cfg']; SZ = lambda e: cfg_of(e)['imgsz']
print('BEST YOLO:', BEST, '| SECOND:', SECOND)
INF = [('imgsz960', 960, {}), ('imgsz1280', 1280, {}), ('nms0.5', SZ(BEST), {'iou': 0.5}), ('nms0.7', SZ(BEST), {'iou': 0.7}),
       ('tta', SZ(BEST), {'augment': True}), ('tta1280', 1280, {'augment': True, 'bs': 2})]
for name, sz, kw in INF:
    for k in RUN_FOLDS:
        if not done(f'I_{BEST}_{name}_f{k}'):
            save_result(f'I_{BEST}_{name}_f{k}', evaluate(yolo_pred(YOLO(str(Wt(BEST, k))), sz, **kw), VAL(k)), dict(weights=BEST, imgsz=sz, fold=k, **kw))
tab = summary(); CAND = {n: (sz, kw) for n, sz, kw in INF}; CAND['native'] = (SZ(BEST), {})
score = {n: tab.loc[BEST if n == 'native' else f'I_{BEST}_{n}', 'mAP'] for n in CAND}; INF_BEST = max(score, key=score.get)
print(pd.Series(score).round(4).to_string()); print('кращий інференс:', INF_BEST, CAND[INF_BEST])

In [ ]:
# 4.4 ансамблі WBF на кожному фолді (OOF): E1 = BEST + SECOND (дві YOLO-конфігурації), E2 = BEST + кращий RT-DETR
from ensemble_boxes import weighted_boxes_fusion
def wbf_pred(fns, weights=None, iou_thr=0.6, skip=0.01):
    def fn(paths):
        outs = [f(paths)[0] for f in fns]; res = {}
        for p in paths:
            kk = str(Path(p)); W, H = Image.open(p).size; bl, sl, ll = [], [], []
            for o in outs:
                x, s, c = o[kk]; bl.append(np.clip(x / [W, H, W, H], 0, 1).tolist()); sl.append(s.tolist()); ll.append(c.tolist())
            b, s, l = weighted_boxes_fusion(bl, sl, ll, weights=weights, iou_thr=iou_thr, skip_box_thr=skip)
            res[kk] = (b * [W, H, W, H], s, l.astype(int))
        return res, float('nan')
    return fn

tab = summary(); DC = [e for e in tab.index if e.startswith('D')]; DB = max(DC, key=lambda e: tab.loc[e, 'mAP']); print('BEST RT-DETR:', DB)
szb, kwb = CAND[INF_BEST]
for k in RUN_FOLDS:
    if BEST != SECOND and not done(f'E1_wbf_best_second_f{k}'):
        ens = wbf_pred([yolo_pred(YOLO(str(Wt(BEST, k))), szb, **kwb), yolo_pred(YOLO(str(Wt(SECOND, k))), SZ(SECOND))])
        save_result(f'E1_wbf_best_second_f{k}', evaluate(ens, VAL(k)), dict(members=[BEST, SECOND], iou_thr=0.6, fold=k))
    if not done(f'E2_wbf_yolo_detr_f{k}'):
        ens = wbf_pred([yolo_pred(YOLO(str(Wt(BEST, k))), szb, **kwb), yolo_pred(RTDETR(str(Wt(DB, k))), SZ(DB))])
        save_result(f'E2_wbf_yolo_detr_f{k}', evaluate(ens, VAL(k)), dict(members=[BEST, DB], iou_thr=0.6, fold=k))
print(summary().to_string())

In [ ]:
# 4.5 S1: YOLO-only. WBF ансамбль fold-моделей кращого YOLO з кращим інференсом (кожна модель бачила 2/3 даних, разом усе). conf=0.001.
S1 = [yolo_pred(YOLO(str(Wt(BEST, k))), szb, conf=0.001, **kwb) for k in RUN_FOLDS]
make_submission(wbf_pred(S1, skip=0.001), SUBS / 'S1_yolo_folds_wbf.csv')

In [ ]:
# 4.5 S2: YOLO + RT-DETR (fold-моделі обох). Обирай S1/S2 за OOF: порівняй E2 з BEST та E1 у таблиці вище.
S2 = S1 + [yolo_pred(RTDETR(str(Wt(DB, k))), SZ(DB), conf=0.001) for k in RUN_FOLDS]
make_submission(wbf_pred(S2, skip=0.001), SUBS / 'S2_yolo_detr_folds_wbf.csv')